# Chapter 90 - A Minimal Generation Script

A command-line program starts without notebook variables and must reconstruct every dependency from code, arguments, and saved files.

This chapter writes a standalone `generate.py`, launches it in fresh Python processes, and tests both successful and failing commands.

## Learning goals

By the end of this chapter, you will be able to:

- explain why a script cannot depend on notebook memory;

- build a self-contained `generate.py` entry point;

- parse prompt, generation, seed, and checkpoint-directory arguments;

- validate JSON and load model state safely on CPU;

- preserve complete long prompts while respecting model context length;

- keep normal standard output limited to generated text;

- test help, deterministic generation, and error paths in subprocesses; and

- identify when one-file simplicity should give way to imported project modules.

## Notebook memory versus process state

A notebook kernel may already contain `model`, `tokenizer`, and config variables from earlier cells.

Every invocation of `python generate.py ...` starts a new process with none of those objects.

The script must therefore define or import model and tokenizer code, parse arguments, read package files, construct objects, load state, and run inference by itself.

Subprocess tests in this chapter enforce that boundary because they cannot see notebook variables.

## Keep the interface minimal

The script supports:

- required `--prompt`;

- `--max-new-tokens`, defaulting to `100`;

- positive `--temperature`, defaulting to `1.0`;

- optional positive `--top-k`;

- optional `--seed`; and

- `--checkpoint-path`, defaulting to `saved_tiny_gpt`.

The checkpoint path names a Chapter 88 directory containing `model_state.pt`, `tokenizer.json`, and `model_config.json`.

Supporting a second bundled-checkpoint schema would add branching and ambiguity without improving this minimal workflow.

## Design corrections carried forward

The script uses the current `TinyGPT-v1` architecture and `model_state.pt` filename rather than stale names from older drafts.

It validates versioned JSON, uses strict `load_state_dict`, loads tensors with `map_location='cpu'` and `weights_only=True`, and calls `model.eval()`.

Sampling receives an explicit `torch.Generator` instead of resetting global random state.

When a prompt exceeds context length, only its suffix conditions generation, but the complete prompt is retained in printed text.

## Prepare an isolated workspace

The notebook writes both the script and a trained package under one temporary directory.

Nothing generated by the demonstration remains in the repository after cleanup.

In [1]:
import tempfile  # noqa: I001
from pathlib import Path


temporary_directory = tempfile.TemporaryDirectory()
temporary_root = Path(temporary_directory.name)
script_path = temporary_root / "generate.py"
checkpoint_directory = temporary_root / "saved_tiny_gpt"
checkpoint_directory.mkdir()

print("Script filename:", script_path.name)
print("Checkpoint directory name:", checkpoint_directory.name)

Script filename: generate.py
Checkpoint directory name: saved_tiny_gpt


## Create a matching package fixture

This producer is setup for the standalone chapter, not part of `generate.py`.

All trained objects remain local to the function, which writes only the same three inference files expected by the script.

In [2]:
class SaveableCharacterTokenizer:
    token_to_id: dict[str, int]
    id_to_token: dict[int, str]
    is_trained: bool

    def __init__(self) -> None:
        self.token_to_id = {}
        self.id_to_token = {}
        self.is_trained = False

    def train(self, text: str) -> None:
        if not text:
            raise ValueError("Training text must not be empty.")
        self.token_to_id = {
            token: token_id for token_id, token in enumerate(sorted(set(text)))
        }
        self.id_to_token = {
            token_id: token for token, token_id in self.token_to_id.items()
        }
        self.is_trained = True

    def _check_trained(self) -> None:
        if not self.is_trained:
            raise RuntimeError("Tokenizer must be trained before use.")

    @property
    def vocabulary_size(self) -> int:
        self._check_trained()
        return len(self.token_to_id)

    def encode(self, text: str) -> list[int]:
        self._check_trained()
        unknown_tokens = sorted(set(text) - set(self.token_to_id))
        if unknown_tokens:
            raise ValueError(f"Unknown tokens: {unknown_tokens!r}.")
        return [self.token_to_id[token] for token in text]

    def decode(self, token_ids: list[int]) -> str:
        self._check_trained()
        return "".join(self.id_to_token[token_id] for token_id in token_ids)

    def to_dict(self) -> dict[str, object]:
        self._check_trained()
        return {
            "format_version": 1,
            "tokenizer_type": "character",
            "token_to_id": self.token_to_id,
        }

    @classmethod
    def from_dict(
        cls,
        tokenizer_data: dict[str, object],
    ) -> "SaveableCharacterTokenizer":
        expected_keys = {"format_version", "tokenizer_type", "token_to_id"}
        if set(tokenizer_data) != expected_keys:
            raise ValueError("Tokenizer JSON has missing or unexpected fields.")
        if tokenizer_data["format_version"] != 1:
            raise ValueError("Unsupported tokenizer format version.")
        if tokenizer_data["tokenizer_type"] != "character":
            raise ValueError("Expected a character tokenizer.")
        raw_mapping = tokenizer_data["token_to_id"]
        if not isinstance(raw_mapping, dict) or not raw_mapping:
            raise TypeError("token_to_id must be a non-empty dictionary.")

        token_to_id: dict[str, int] = {}
        for token, token_id in raw_mapping.items():
            if not isinstance(token, str) or type(token_id) is not int:
                raise TypeError("Tokenizer entries must map strings to integer IDs.")
            token_to_id[token] = token_id
        expected_ids = list(range(len(token_to_id)))
        if sorted(token_to_id.values()) != expected_ids:
            raise ValueError("Tokenizer IDs must be unique and contiguous from zero.")

        tokenizer = cls()
        tokenizer.token_to_id = token_to_id
        tokenizer.id_to_token = {
            token_id: token for token, token_id in token_to_id.items()
        }
        tokenizer.is_trained = True
        return tokenizer

In [3]:
import math  # noqa: I001
import torch


class MultiHeadCausalSelfAttention(torch.nn.Module):
    embedding_dimension: int
    number_of_attention_heads: int
    head_size: int

    def __init__(
        self,
        embedding_dimension: int,
        number_of_attention_heads: int,
        context_length: int,
        dropout_rate: float,
    ) -> None:
        super().__init__()
        if embedding_dimension % number_of_attention_heads != 0:
            raise ValueError("Attention heads must divide the embedding dimension.")
        self.embedding_dimension = embedding_dimension
        self.number_of_attention_heads = number_of_attention_heads
        self.head_size = embedding_dimension // number_of_attention_heads
        self.query_projection = torch.nn.Linear(
            embedding_dimension, embedding_dimension, bias=False
        )
        self.key_projection = torch.nn.Linear(
            embedding_dimension, embedding_dimension, bias=False
        )
        self.value_projection = torch.nn.Linear(
            embedding_dimension, embedding_dimension, bias=False
        )
        self.output_projection = torch.nn.Linear(
            embedding_dimension, embedding_dimension
        )
        self.attention_dropout = torch.nn.Dropout(dropout_rate)
        self.output_dropout = torch.nn.Dropout(dropout_rate)
        self.register_buffer(
            "causal_mask",
            torch.tril(torch.ones(context_length, context_length, dtype=torch.bool)),
        )

    def forward(self, input_values: torch.Tensor) -> torch.Tensor:
        batch_size, sequence_length, _ = input_values.shape

        def split_heads(values: torch.Tensor) -> torch.Tensor:
            return values.view(
                batch_size,
                sequence_length,
                self.number_of_attention_heads,
                self.head_size,
            ).transpose(1, 2)

        queries = split_heads(self.query_projection(input_values))
        keys = split_heads(self.key_projection(input_values))
        values = split_heads(self.value_projection(input_values))
        attention_scores = queries @ keys.transpose(-2, -1)
        attention_scores = attention_scores / math.sqrt(self.head_size)
        causal_mask = self.causal_mask[:sequence_length, :sequence_length]
        attention_scores = attention_scores.masked_fill(~causal_mask, float("-inf"))
        attention_weights = torch.softmax(attention_scores, dim=-1)
        attended_values = self.attention_dropout(attention_weights) @ values
        concatenated_values = (
            attended_values.transpose(1, 2)
            .contiguous()
            .view(batch_size, sequence_length, self.embedding_dimension)
        )
        projected_values = self.output_projection(concatenated_values)
        output_values: torch.Tensor = self.output_dropout(projected_values)
        return output_values

In [4]:
class FeedForwardNetwork(torch.nn.Module):
    def __init__(self, embedding_dimension: int, dropout_rate: float) -> None:
        super().__init__()
        self.network = torch.nn.Sequential(
            torch.nn.Linear(embedding_dimension, 4 * embedding_dimension),
            torch.nn.ReLU(),
            torch.nn.Linear(4 * embedding_dimension, embedding_dimension),
            torch.nn.Dropout(dropout_rate),
        )

    def forward(self, input_values: torch.Tensor) -> torch.Tensor:
        output_values: torch.Tensor = self.network(input_values)
        return output_values


class TransformerBlock(torch.nn.Module):
    def __init__(
        self,
        embedding_dimension: int,
        number_of_attention_heads: int,
        context_length: int,
        dropout_rate: float,
    ) -> None:
        super().__init__()
        self.attention_norm = torch.nn.LayerNorm(embedding_dimension)
        self.attention = MultiHeadCausalSelfAttention(
            embedding_dimension=embedding_dimension,
            number_of_attention_heads=number_of_attention_heads,
            context_length=context_length,
            dropout_rate=dropout_rate,
        )
        self.feedforward_norm = torch.nn.LayerNorm(embedding_dimension)
        self.feedforward = FeedForwardNetwork(
            embedding_dimension=embedding_dimension,
            dropout_rate=dropout_rate,
        )

    def forward(self, input_values: torch.Tensor) -> torch.Tensor:
        attention_branch = self.attention(self.attention_norm(input_values))
        values_after_attention = input_values + attention_branch
        feedforward_branch = self.feedforward(
            self.feedforward_norm(values_after_attention)
        )
        output_values: torch.Tensor = values_after_attention + feedforward_branch
        return output_values


class TinyGPT(torch.nn.Module):
    vocabulary_size: int
    context_length: int

    def __init__(
        self,
        vocabulary_size: int,
        context_length: int,
        embedding_dimension: int,
        number_of_attention_heads: int,
        number_of_transformer_blocks: int,
        dropout_rate: float,
    ) -> None:
        super().__init__()
        self.vocabulary_size = vocabulary_size
        self.context_length = context_length
        self.token_embedding = torch.nn.Embedding(vocabulary_size, embedding_dimension)
        self.position_embedding = torch.nn.Embedding(
            context_length, embedding_dimension
        )
        self.embedding_dropout = torch.nn.Dropout(dropout_rate)
        self.transformer_blocks = torch.nn.Sequential(
            *[
                TransformerBlock(
                    embedding_dimension=embedding_dimension,
                    number_of_attention_heads=number_of_attention_heads,
                    context_length=context_length,
                    dropout_rate=dropout_rate,
                )
                for _ in range(number_of_transformer_blocks)
            ]
        )
        self.final_norm = torch.nn.LayerNorm(embedding_dimension)
        self.output_layer = torch.nn.Linear(embedding_dimension, vocabulary_size)

    def forward(
        self,
        input_token_ids: torch.Tensor,
        target_token_ids: torch.Tensor | None = None,
    ) -> tuple[torch.Tensor, torch.Tensor | None]:
        if input_token_ids.ndim != 2:
            raise ValueError("Inputs must have shape [batch, time].")
        batch_size, sequence_length = input_token_ids.shape
        if sequence_length > self.context_length:
            raise ValueError("Input exceeds model context length.")
        position_ids = torch.arange(sequence_length, device=input_token_ids.device)
        hidden_values = self.token_embedding(input_token_ids)
        hidden_values = hidden_values + self.position_embedding(position_ids)
        hidden_values = self.embedding_dropout(hidden_values)
        hidden_values = self.transformer_blocks(hidden_values)
        logits = self.output_layer(self.final_norm(hidden_values))
        loss = None
        if target_token_ids is not None:
            if target_token_ids.shape != input_token_ids.shape:
                raise ValueError("Targets must have the same shape as inputs.")
            loss = torch.nn.functional.cross_entropy(
                logits.reshape(batch_size * sequence_length, self.vocabulary_size),
                target_token_ids.reshape(batch_size * sequence_length),
            )
        return logits, loss

    @torch.no_grad()
    def generate(
        self,
        input_token_ids: torch.Tensor,
        number_of_new_tokens: int,
        generator: torch.Generator,
        temperature: float = 1.0,
        top_k: int | None = None,
    ) -> torch.Tensor:
        if temperature <= 0:
            raise ValueError("temperature must be positive.")
        generated_ids = input_token_ids
        for _ in range(number_of_new_tokens):
            model_input = generated_ids[:, -self.context_length :]
            logits, _ = self(model_input)
            next_logits = logits[:, -1] / temperature
            if top_k is not None:
                effective_top_k = min(top_k, self.vocabulary_size)
                top_values, _ = torch.topk(next_logits, effective_top_k)
                next_logits = next_logits.masked_fill(
                    next_logits < top_values[:, -1, None], float("-inf")
                )
            probabilities = torch.softmax(next_logits, dim=-1)
            next_ids = torch.multinomial(
                probabilities, num_samples=1, generator=generator
            )
            generated_ids = torch.cat([generated_ids, next_ids], dim=1)
        return generated_ids

In [5]:
from dataclasses import dataclass


def require_int(
    data: dict[str, object],
    key: str,
    minimum: int,
) -> int:
    value = data.get(key)
    if type(value) is not int or value < minimum:
        raise ValueError(f"{key} must be an integer of at least {minimum}.")
    return value


def require_float(
    data: dict[str, object],
    key: str,
    minimum: float,
    maximum: float | None = None,
) -> float:
    value = data.get(key)
    if type(value) is not float or value < minimum:
        raise ValueError(f"{key} must be a floating-point value of at least {minimum}.")
    if maximum is not None and value >= maximum:
        raise ValueError(f"{key} must be less than {maximum}.")
    return value


@dataclass(frozen=True)
class ModelConfig:
    format_version: int
    architecture: str
    vocabulary_size: int
    context_length: int
    embedding_dimension: int
    number_of_attention_heads: int
    number_of_transformer_blocks: int
    dropout_rate: float

    def to_dict(self) -> dict[str, object]:
        return {
            "format_version": self.format_version,
            "architecture": self.architecture,
            "vocabulary_size": self.vocabulary_size,
            "context_length": self.context_length,
            "embedding_dimension": self.embedding_dimension,
            "number_of_attention_heads": self.number_of_attention_heads,
            "number_of_transformer_blocks": self.number_of_transformer_blocks,
            "dropout_rate": self.dropout_rate,
        }

    @classmethod
    def from_dict(cls, data: dict[str, object]) -> "ModelConfig":
        expected_keys = {
            "format_version",
            "architecture",
            "vocabulary_size",
            "context_length",
            "embedding_dimension",
            "number_of_attention_heads",
            "number_of_transformer_blocks",
            "dropout_rate",
        }
        if set(data) != expected_keys:
            raise ValueError("Model config has missing or unexpected fields.")
        if data["format_version"] != 1 or data["architecture"] != "TinyGPT-v1":
            raise ValueError("Unsupported model config format or architecture.")
        config = cls(
            format_version=1,
            architecture="TinyGPT-v1",
            vocabulary_size=require_int(data, "vocabulary_size", 1),
            context_length=require_int(data, "context_length", 1),
            embedding_dimension=require_int(data, "embedding_dimension", 1),
            number_of_attention_heads=require_int(data, "number_of_attention_heads", 1),
            number_of_transformer_blocks=require_int(
                data, "number_of_transformer_blocks", 1
            ),
            dropout_rate=require_float(data, "dropout_rate", 0.0, 1.0),
        )
        if config.embedding_dimension % config.number_of_attention_heads != 0:
            raise ValueError("Attention heads must divide the embedding dimension.")
        return config


@dataclass(frozen=True)
class TrainingConfig:
    format_version: int
    batch_size: int
    learning_rate: float
    training_steps: int
    weight_decay: float
    max_gradient_norm: float
    model_seed: int
    training_seed: int
    dataset_description: str

    def to_dict(self) -> dict[str, object]:
        return {
            "format_version": self.format_version,
            "batch_size": self.batch_size,
            "learning_rate": self.learning_rate,
            "training_steps": self.training_steps,
            "weight_decay": self.weight_decay,
            "max_gradient_norm": self.max_gradient_norm,
            "model_seed": self.model_seed,
            "training_seed": self.training_seed,
            "dataset_description": self.dataset_description,
        }

    @classmethod
    def from_dict(cls, data: dict[str, object]) -> "TrainingConfig":
        expected_keys = {
            "format_version",
            "batch_size",
            "learning_rate",
            "training_steps",
            "weight_decay",
            "max_gradient_norm",
            "model_seed",
            "training_seed",
            "dataset_description",
        }
        if set(data) != expected_keys or data["format_version"] != 1:
            raise ValueError("Unsupported training config schema.")
        description = data["dataset_description"]
        if not isinstance(description, str) or not description:
            raise ValueError("dataset_description must be a non-empty string.")
        learning_rate = require_float(data, "learning_rate", 0.0)
        max_gradient_norm = require_float(data, "max_gradient_norm", 0.0)
        if learning_rate == 0.0 or max_gradient_norm == 0.0:
            raise ValueError("learning_rate and max_gradient_norm must be positive.")
        return cls(
            format_version=1,
            batch_size=require_int(data, "batch_size", 1),
            learning_rate=learning_rate,
            training_steps=require_int(data, "training_steps", 1),
            weight_decay=require_float(data, "weight_decay", 0.0),
            max_gradient_norm=max_gradient_norm,
            model_seed=require_int(data, "model_seed", 0),
            training_seed=require_int(data, "training_seed", 0),
            dataset_description=description,
        )

In [6]:
import json  # noqa: I001
from collections.abc import Mapping


def save_json(data: Mapping[str, object], path: Path) -> None:
    serialized = json.dumps(dict(data), indent=2, ensure_ascii=False, sort_keys=True)
    path.write_text(serialized + "\n", encoding="utf-8")


def create_model(config: ModelConfig) -> TinyGPT:
    return TinyGPT(
        vocabulary_size=config.vocabulary_size,
        context_length=config.context_length,
        embedding_dimension=config.embedding_dimension,
        number_of_attention_heads=config.number_of_attention_heads,
        number_of_transformer_blocks=config.number_of_transformer_blocks,
        dropout_rate=config.dropout_rate,
    )


def create_saved_package_fixture(artifact_directory: Path) -> list[float]:
    training_text = (
        "Alice saw the rabbit. "
        "The rabbit saw Alice. "
        "Alice followed the rabbit. "
        "The rabbit was late. "
    ) * 40
    tokenizer = SaveableCharacterTokenizer()
    tokenizer.train(training_text)
    model_config = ModelConfig(
        format_version=1,
        architecture="TinyGPT-v1",
        vocabulary_size=tokenizer.vocabulary_size,
        context_length=64,
        embedding_dimension=64,
        number_of_attention_heads=4,
        number_of_transformer_blocks=2,
        dropout_rate=0.1,
    )
    torch.manual_seed(9001)
    model = create_model(model_config)
    optimizer = torch.optim.AdamW(model.parameters(), lr=0.0003, weight_decay=0.01)
    token_ids = torch.tensor(tokenizer.encode(training_text), dtype=torch.long)
    batch_generator = torch.Generator().manual_seed(9002)
    torch.manual_seed(9002)
    losses: list[float] = []
    model.train()
    for step in range(1, 301):
        starts = torch.randint(
            0,
            token_ids.numel() - model_config.context_length,
            (8,),
            generator=batch_generator,
        )
        indexes = starts[:, None] + torch.arange(model_config.context_length)[None, :]
        _, loss = model(token_ids[indexes], token_ids[indexes + 1])
        if loss is None or not torch.isfinite(loss):
            raise RuntimeError("Fixture training loss is non-finite.")
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(
            model.parameters(), max_norm=1.0, error_if_nonfinite=True
        )
        optimizer.step()
        if step == 1 or step % 100 == 0:
            losses.append(float(loss.item()))

    torch.save(model.state_dict(), artifact_directory / "model_state.pt")
    save_json(tokenizer.to_dict(), artifact_directory / "tokenizer.json")
    save_json(model_config.to_dict(), artifact_directory / "model_config.json")
    return losses


fixture_losses = create_saved_package_fixture(checkpoint_directory)
print("Fixture batch losses:", [round(loss, 4) for loss in fixture_losses])
print("Package files:", sorted(path.name for path in checkpoint_directory.iterdir()))
assert fixture_losses[-1] < fixture_losses[0]

Fixture batch losses: [2.9603, 1.2944, 0.971, 0.8601]
Package files: ['model_config.json', 'model_state.pt', 'tokenizer.json']


The falling batch loss confirms that command-line tests use learned model state rather than an untrained random fixture.

## Write the complete `generate.py`

The string below is the complete script source.

It contains no references to notebook-only variables or paths.

In [7]:
generation_script = r"""
import argparse
import json
import math
import sys
from collections.abc import Sequence
from dataclasses import dataclass
from pathlib import Path
from typing import cast

import torch


class SaveableCharacterTokenizer:
    token_to_id: dict[str, int]
    id_to_token: dict[int, str]
    is_trained: bool

    def __init__(self) -> None:
        self.token_to_id = {}
        self.id_to_token = {}
        self.is_trained = False

    def train(self, text: str) -> None:
        if not text:
            raise ValueError("Training text must not be empty.")
        self.token_to_id = {
            token: token_id for token_id, token in enumerate(sorted(set(text)))
        }
        self.id_to_token = {
            token_id: token for token, token_id in self.token_to_id.items()
        }
        self.is_trained = True

    def _check_trained(self) -> None:
        if not self.is_trained:
            raise RuntimeError("Tokenizer must be trained before use.")

    @property
    def vocabulary_size(self) -> int:
        self._check_trained()
        return len(self.token_to_id)

    def encode(self, text: str) -> list[int]:
        self._check_trained()
        unknown_tokens = sorted(set(text) - set(self.token_to_id))
        if unknown_tokens:
            raise ValueError(f"Unknown tokens: {unknown_tokens!r}.")
        return [self.token_to_id[token] for token in text]

    def decode(self, token_ids: list[int]) -> str:
        self._check_trained()
        return "".join(self.id_to_token[token_id] for token_id in token_ids)

    def to_dict(self) -> dict[str, object]:
        self._check_trained()
        return {
            "format_version": 1,
            "tokenizer_type": "character",
            "token_to_id": self.token_to_id,
        }

    @classmethod
    def from_dict(
        cls,
        tokenizer_data: dict[str, object],
    ) -> "SaveableCharacterTokenizer":
        expected_keys = {"format_version", "tokenizer_type", "token_to_id"}
        if set(tokenizer_data) != expected_keys:
            raise ValueError("Tokenizer JSON has missing or unexpected fields.")
        if tokenizer_data["format_version"] != 1:
            raise ValueError("Unsupported tokenizer format version.")
        if tokenizer_data["tokenizer_type"] != "character":
            raise ValueError("Expected a character tokenizer.")
        raw_mapping = tokenizer_data["token_to_id"]
        if not isinstance(raw_mapping, dict) or not raw_mapping:
            raise TypeError("token_to_id must be a non-empty dictionary.")

        token_to_id: dict[str, int] = {}
        for token, token_id in raw_mapping.items():
            if not isinstance(token, str) or type(token_id) is not int:
                raise TypeError("Tokenizer entries must map strings to integer IDs.")
            token_to_id[token] = token_id
        expected_ids = list(range(len(token_to_id)))
        if sorted(token_to_id.values()) != expected_ids:
            raise ValueError("Tokenizer IDs must be unique and contiguous from zero.")

        tokenizer = cls()
        tokenizer.token_to_id = token_to_id
        tokenizer.id_to_token = {
            token_id: token for token, token_id in token_to_id.items()
        }
        tokenizer.is_trained = True
        return tokenizer


class MultiHeadCausalSelfAttention(torch.nn.Module):
    embedding_dimension: int
    number_of_attention_heads: int
    head_size: int

    def __init__(
        self,
        embedding_dimension: int,
        number_of_attention_heads: int,
        context_length: int,
        dropout_rate: float,
    ) -> None:
        super().__init__()
        if embedding_dimension % number_of_attention_heads != 0:
            raise ValueError("Attention heads must divide the embedding dimension.")
        self.embedding_dimension = embedding_dimension
        self.number_of_attention_heads = number_of_attention_heads
        self.head_size = embedding_dimension // number_of_attention_heads
        self.query_projection = torch.nn.Linear(
            embedding_dimension, embedding_dimension, bias=False
        )
        self.key_projection = torch.nn.Linear(
            embedding_dimension, embedding_dimension, bias=False
        )
        self.value_projection = torch.nn.Linear(
            embedding_dimension, embedding_dimension, bias=False
        )
        self.output_projection = torch.nn.Linear(
            embedding_dimension, embedding_dimension
        )
        self.attention_dropout = torch.nn.Dropout(dropout_rate)
        self.output_dropout = torch.nn.Dropout(dropout_rate)
        self.register_buffer(
            "causal_mask",
            torch.tril(torch.ones(context_length, context_length, dtype=torch.bool)),
        )

    def forward(self, input_values: torch.Tensor) -> torch.Tensor:
        batch_size, sequence_length, _ = input_values.shape

        def split_heads(values: torch.Tensor) -> torch.Tensor:
            return values.view(
                batch_size,
                sequence_length,
                self.number_of_attention_heads,
                self.head_size,
            ).transpose(1, 2)

        queries = split_heads(self.query_projection(input_values))
        keys = split_heads(self.key_projection(input_values))
        values = split_heads(self.value_projection(input_values))
        attention_scores = queries @ keys.transpose(-2, -1)
        attention_scores = attention_scores / math.sqrt(self.head_size)
        causal_mask = self.causal_mask[:sequence_length, :sequence_length]
        attention_scores = attention_scores.masked_fill(~causal_mask, float("-inf"))
        attention_weights = torch.softmax(attention_scores, dim=-1)
        attended_values = self.attention_dropout(attention_weights) @ values
        concatenated_values = (
            attended_values.transpose(1, 2)
            .contiguous()
            .view(batch_size, sequence_length, self.embedding_dimension)
        )
        projected_values = self.output_projection(concatenated_values)
        output_values: torch.Tensor = self.output_dropout(projected_values)
        return output_values


class FeedForwardNetwork(torch.nn.Module):
    def __init__(self, embedding_dimension: int, dropout_rate: float) -> None:
        super().__init__()
        self.network = torch.nn.Sequential(
            torch.nn.Linear(embedding_dimension, 4 * embedding_dimension),
            torch.nn.ReLU(),
            torch.nn.Linear(4 * embedding_dimension, embedding_dimension),
            torch.nn.Dropout(dropout_rate),
        )

    def forward(self, input_values: torch.Tensor) -> torch.Tensor:
        output_values: torch.Tensor = self.network(input_values)
        return output_values


class TransformerBlock(torch.nn.Module):
    def __init__(
        self,
        embedding_dimension: int,
        number_of_attention_heads: int,
        context_length: int,
        dropout_rate: float,
    ) -> None:
        super().__init__()
        self.attention_norm = torch.nn.LayerNorm(embedding_dimension)
        self.attention = MultiHeadCausalSelfAttention(
            embedding_dimension=embedding_dimension,
            number_of_attention_heads=number_of_attention_heads,
            context_length=context_length,
            dropout_rate=dropout_rate,
        )
        self.feedforward_norm = torch.nn.LayerNorm(embedding_dimension)
        self.feedforward = FeedForwardNetwork(
            embedding_dimension=embedding_dimension,
            dropout_rate=dropout_rate,
        )

    def forward(self, input_values: torch.Tensor) -> torch.Tensor:
        attention_branch = self.attention(self.attention_norm(input_values))
        values_after_attention = input_values + attention_branch
        feedforward_branch = self.feedforward(
            self.feedforward_norm(values_after_attention)
        )
        output_values: torch.Tensor = values_after_attention + feedforward_branch
        return output_values


class TinyGPT(torch.nn.Module):
    vocabulary_size: int
    context_length: int

    def __init__(
        self,
        vocabulary_size: int,
        context_length: int,
        embedding_dimension: int,
        number_of_attention_heads: int,
        number_of_transformer_blocks: int,
        dropout_rate: float,
    ) -> None:
        super().__init__()
        self.vocabulary_size = vocabulary_size
        self.context_length = context_length
        self.token_embedding = torch.nn.Embedding(vocabulary_size, embedding_dimension)
        self.position_embedding = torch.nn.Embedding(
            context_length, embedding_dimension
        )
        self.embedding_dropout = torch.nn.Dropout(dropout_rate)
        self.transformer_blocks = torch.nn.Sequential(
            *[
                TransformerBlock(
                    embedding_dimension=embedding_dimension,
                    number_of_attention_heads=number_of_attention_heads,
                    context_length=context_length,
                    dropout_rate=dropout_rate,
                )
                for _ in range(number_of_transformer_blocks)
            ]
        )
        self.final_norm = torch.nn.LayerNorm(embedding_dimension)
        self.output_layer = torch.nn.Linear(embedding_dimension, vocabulary_size)

    def forward(
        self,
        input_token_ids: torch.Tensor,
        target_token_ids: torch.Tensor | None = None,
    ) -> tuple[torch.Tensor, torch.Tensor | None]:
        if input_token_ids.ndim != 2:
            raise ValueError("Inputs must have shape [batch, time].")
        batch_size, sequence_length = input_token_ids.shape
        if sequence_length > self.context_length:
            raise ValueError("Input exceeds model context length.")
        position_ids = torch.arange(sequence_length, device=input_token_ids.device)
        hidden_values = self.token_embedding(input_token_ids)
        hidden_values = hidden_values + self.position_embedding(position_ids)
        hidden_values = self.embedding_dropout(hidden_values)
        hidden_values = self.transformer_blocks(hidden_values)
        logits = self.output_layer(self.final_norm(hidden_values))
        loss = None
        if target_token_ids is not None:
            if target_token_ids.shape != input_token_ids.shape:
                raise ValueError("Targets must have the same shape as inputs.")
            loss = torch.nn.functional.cross_entropy(
                logits.reshape(batch_size * sequence_length, self.vocabulary_size),
                target_token_ids.reshape(batch_size * sequence_length),
            )
        return logits, loss

    @torch.no_grad()
    def generate(
        self,
        input_token_ids: torch.Tensor,
        number_of_new_tokens: int,
        generator: torch.Generator,
        temperature: float = 1.0,
        top_k: int | None = None,
    ) -> torch.Tensor:
        if temperature <= 0:
            raise ValueError("temperature must be positive.")
        generated_ids = input_token_ids
        for _ in range(number_of_new_tokens):
            model_input = generated_ids[:, -self.context_length :]
            logits, _ = self(model_input)
            next_logits = logits[:, -1] / temperature
            if top_k is not None:
                effective_top_k = min(top_k, self.vocabulary_size)
                top_values, _ = torch.topk(next_logits, effective_top_k)
                next_logits = next_logits.masked_fill(
                    next_logits < top_values[:, -1, None], float("-inf")
                )
            probabilities = torch.softmax(next_logits, dim=-1)
            next_ids = torch.multinomial(
                probabilities, num_samples=1, generator=generator
            )
            generated_ids = torch.cat([generated_ids, next_ids], dim=1)
        return generated_ids


def require_int(
    data: dict[str, object],
    key: str,
    minimum: int,
) -> int:
    value = data.get(key)
    if type(value) is not int or value < minimum:
        raise ValueError(f"{key} must be an integer of at least {minimum}.")
    return value


def require_float(
    data: dict[str, object],
    key: str,
    minimum: float,
    maximum: float | None = None,
) -> float:
    value = data.get(key)
    if type(value) is not float or value < minimum:
        raise ValueError(f"{key} must be a floating-point value of at least {minimum}.")
    if maximum is not None and value >= maximum:
        raise ValueError(f"{key} must be less than {maximum}.")
    return value


@dataclass(frozen=True)
class ModelConfig:
    format_version: int
    architecture: str
    vocabulary_size: int
    context_length: int
    embedding_dimension: int
    number_of_attention_heads: int
    number_of_transformer_blocks: int
    dropout_rate: float

    def to_dict(self) -> dict[str, object]:
        return {
            "format_version": self.format_version,
            "architecture": self.architecture,
            "vocabulary_size": self.vocabulary_size,
            "context_length": self.context_length,
            "embedding_dimension": self.embedding_dimension,
            "number_of_attention_heads": self.number_of_attention_heads,
            "number_of_transformer_blocks": self.number_of_transformer_blocks,
            "dropout_rate": self.dropout_rate,
        }

    @classmethod
    def from_dict(cls, data: dict[str, object]) -> "ModelConfig":
        expected_keys = {
            "format_version",
            "architecture",
            "vocabulary_size",
            "context_length",
            "embedding_dimension",
            "number_of_attention_heads",
            "number_of_transformer_blocks",
            "dropout_rate",
        }
        if set(data) != expected_keys:
            raise ValueError("Model config has missing or unexpected fields.")
        if data["format_version"] != 1 or data["architecture"] != "TinyGPT-v1":
            raise ValueError("Unsupported model config format or architecture.")
        config = cls(
            format_version=1,
            architecture="TinyGPT-v1",
            vocabulary_size=require_int(data, "vocabulary_size", 1),
            context_length=require_int(data, "context_length", 1),
            embedding_dimension=require_int(data, "embedding_dimension", 1),
            number_of_attention_heads=require_int(data, "number_of_attention_heads", 1),
            number_of_transformer_blocks=require_int(
                data, "number_of_transformer_blocks", 1
            ),
            dropout_rate=require_float(data, "dropout_rate", 0.0, 1.0),
        )
        if config.embedding_dimension % config.number_of_attention_heads != 0:
            raise ValueError("Attention heads must divide the embedding dimension.")
        return config


def create_model(config: ModelConfig) -> TinyGPT:
    return TinyGPT(
        vocabulary_size=config.vocabulary_size,
        context_length=config.context_length,
        embedding_dimension=config.embedding_dimension,
        number_of_attention_heads=config.number_of_attention_heads,
        number_of_transformer_blocks=config.number_of_transformer_blocks,
        dropout_rate=config.dropout_rate,
    )


def load_json_object(path: Path) -> dict[str, object]:
    loaded_value = cast(object, json.loads(path.read_text(encoding="utf-8")))
    if not isinstance(loaded_value, dict):
        raise TypeError(f"{path.name} must contain a JSON object.")
    result: dict[str, object] = {}
    for key, value in loaded_value.items():
        if not isinstance(key, str):
            raise TypeError("JSON object keys must be strings.")
        result[key] = value
    return result


def load_tensor_state_dict(path: Path) -> dict[str, torch.Tensor]:
    loaded_value: object = torch.load(
        path,
        map_location="cpu",
        weights_only=True,
    )
    if not isinstance(loaded_value, dict):
        raise TypeError("Model state must be a dictionary.")
    tensor_state: dict[str, torch.Tensor] = {}
    for key, value in loaded_value.items():
        if not isinstance(key, str) or not isinstance(value, torch.Tensor):
            raise TypeError("Model state must map string names to tensors.")
        tensor_state[key] = value
    return tensor_state


def load_saved_system(
    checkpoint_path: Path,
) -> tuple[TinyGPT, SaveableCharacterTokenizer, ModelConfig]:
    required_filenames = ["model_state.pt", "tokenizer.json", "model_config.json"]
    missing_filenames = [
        filename
        for filename in required_filenames
        if not (checkpoint_path / filename).is_file()
    ]
    if missing_filenames:
        raise FileNotFoundError(f"Missing required files: {missing_filenames!r}.")

    tokenizer = SaveableCharacterTokenizer.from_dict(
        load_json_object(checkpoint_path / "tokenizer.json")
    )
    model_config = ModelConfig.from_dict(
        load_json_object(checkpoint_path / "model_config.json")
    )
    if tokenizer.vocabulary_size != model_config.vocabulary_size:
        raise ValueError("Tokenizer and model-config vocabulary sizes differ.")

    model = create_model(model_config)
    state_dictionary = load_tensor_state_dict(checkpoint_path / "model_state.pt")
    try:
        incompatible_keys = model.load_state_dict(state_dictionary, strict=True)
    except RuntimeError as error:
        raise RuntimeError(
            "Model state is incompatible with the saved config and model code."
        ) from error
    if incompatible_keys.missing_keys or incompatible_keys.unexpected_keys:
        raise RuntimeError("Model state did not load strictly.")
    model.eval()
    return model, tokenizer, model_config


def nonnegative_integer(value: str) -> int:
    parsed_value = int(value)
    if parsed_value < 0:
        raise argparse.ArgumentTypeError("value must be nonnegative")
    return parsed_value


def positive_integer(value: str) -> int:
    parsed_value = int(value)
    if parsed_value < 1:
        raise argparse.ArgumentTypeError("value must be at least 1")
    return parsed_value


def positive_float(value: str) -> float:
    parsed_value = float(value)
    if parsed_value <= 0:
        raise argparse.ArgumentTypeError("value must be positive")
    return parsed_value


@dataclass(frozen=True)
class GenerationArguments:
    checkpoint_path: Path
    prompt: str
    max_new_tokens: int
    temperature: float
    top_k: int | None
    seed: int | None


def build_argument_parser() -> argparse.ArgumentParser:
    parser = argparse.ArgumentParser(
        description="Generate text from a saved TinyGPT directory."
    )
    parser.add_argument(
        "--checkpoint-path",
        type=Path,
        default=Path("saved_tiny_gpt"),
        help="Directory containing the saved model package.",
    )
    parser.add_argument(
        "--prompt",
        required=True,
        help="Text used to start generation.",
    )
    parser.add_argument(
        "--max-new-tokens",
        type=nonnegative_integer,
        default=100,
        help="Number of continuation tokens to sample (default: 100).",
    )
    parser.add_argument(
        "--temperature",
        type=positive_float,
        default=1.0,
        help="Positive sampling temperature (default: 1.0).",
    )
    parser.add_argument(
        "--top-k",
        type=positive_integer,
        default=None,
        help="Optional positive top-k sampling limit.",
    )
    parser.add_argument(
        "--seed",
        type=int,
        default=None,
        help="Optional sampling seed for reproducible output.",
    )
    return parser


def parse_arguments(
    argv: Sequence[str] | None = None,
) -> GenerationArguments:
    namespace = build_argument_parser().parse_args(argv)
    return GenerationArguments(
        checkpoint_path=cast(Path, namespace.checkpoint_path),
        prompt=cast(str, namespace.prompt),
        max_new_tokens=cast(int, namespace.max_new_tokens),
        temperature=cast(float, namespace.temperature),
        top_k=cast(int | None, namespace.top_k),
        seed=cast(int | None, namespace.seed),
    )


@torch.inference_mode()
def generate_text(
    model: TinyGPT,
    tokenizer: SaveableCharacterTokenizer,
    prompt: str,
    max_new_tokens: int,
    temperature: float,
    top_k: int | None,
    seed: int | None,
) -> str:
    if not prompt:
        raise ValueError("Prompt must not be empty.")
    model.eval()
    prompt_ids = tokenizer.encode(prompt)
    conditioning_ids = prompt_ids[-model.context_length :]
    input_ids = torch.tensor([conditioning_ids], dtype=torch.long)
    generator = torch.Generator()
    if seed is None:
        generator.seed()
    else:
        generator.manual_seed(seed)
    generated_ids = model.generate(
        input_ids,
        number_of_new_tokens=max_new_tokens,
        generator=generator,
        temperature=temperature,
        top_k=top_k,
    )
    continuation_ids = generated_ids[0, len(conditioning_ids) :].tolist()
    return prompt + tokenizer.decode(continuation_ids)


def main(argv: Sequence[str] | None = None) -> int:
    arguments = parse_arguments(argv)
    try:
        model, tokenizer, _ = load_saved_system(arguments.checkpoint_path)
        generated_text = generate_text(
            model=model,
            tokenizer=tokenizer,
            prompt=arguments.prompt,
            max_new_tokens=arguments.max_new_tokens,
            temperature=arguments.temperature,
            top_k=arguments.top_k,
            seed=arguments.seed,
        )
    except (FileNotFoundError, KeyError, TypeError, ValueError, RuntimeError) as error:
        print(f"error: {error}", file=sys.stderr)
        return 1
    print(generated_text)
    return 0


if __name__ == "__main__":
    raise SystemExit(main())
""".lstrip()

script_path.write_text(generation_script, encoding="utf-8")
compile(generation_script, script_path.name, "exec")
print("Script lines:", len(generation_script.splitlines()))
print("Script compiles:", True)

Script lines: 596
Script compiles: True


The compilation check catches syntax errors before a subprocess attempts to run the file.

The script is long because a one-file program must contain model, tokenizer, schema, loader, CLI, and entry-point code that a larger project would import.

## Launch commands in fresh processes

`subprocess.run` starts the same kind of independent Python process a shell command would use.

The helper captures standard output and standard error separately so normal generation can remain pipe-friendly.

In [8]:
import subprocess  # noqa: I001
import sys
from dataclasses import dataclass


@dataclass(frozen=True)
class CommandResult:
    return_code: int
    standard_output: str
    standard_error: str


def run_generation_script(arguments: list[str]) -> CommandResult:
    completed = subprocess.run(
        [sys.executable, str(script_path), *arguments],
        check=False,
        capture_output=True,
        text=True,
        timeout=60,
    )
    return CommandResult(
        return_code=completed.returncode,
        standard_output=completed.stdout,
        standard_error=completed.stderr,
    )


checkpoint_arguments = ["--checkpoint-path", str(checkpoint_directory)]

## Inspect command-line help

The help command should exit successfully without loading a checkpoint because argument parsing handles `--help` first.

In [9]:
help_result = run_generation_script(["--help"])
help_lines = help_result.standard_output.splitlines()

print("Return code:", help_result.return_code)
print("First help line:", help_lines[0])
print("Documents --prompt:", "--prompt" in help_result.standard_output)
print("Documents --top-k:", "--top-k" in help_result.standard_output)
print("Standard error is empty:", help_result.standard_error == "")

assert help_result.return_code == 0
assert "--prompt" in help_result.standard_output
assert "--checkpoint-path" in help_result.standard_output
assert help_result.standard_error == ""

Return code: 0
First help line: usage: generate.py [-h] [--checkpoint-path CHECKPOINT_PATH] --prompt PROMPT
Documents --prompt: True
Documents --top-k: True
Standard error is empty: True


## Generate from the command line

The successful command passes every supported generation option and prints only generated text to standard output.

In [10]:
generation_arguments = [
    *checkpoint_arguments,
    "--prompt",
    "Alice",
    "--max-new-tokens",
    "80",
    "--temperature",
    "0.8",
    "--top-k",
    "10",
    "--seed",
    "9003",
]
generation_result = run_generation_script(generation_arguments)

print("Return code:", generation_result.return_code)
print("Standard error is empty:", generation_result.standard_error == "")
print("Generated text:")
print(generation_result.standard_output.rstrip("\n"))

assert generation_result.return_code == 0
assert generation_result.standard_error == ""
assert generation_result.standard_output.startswith("Alice")

Return code: 0
Standard error is empty: True
Generated text:
Alice rabice. Aliced sabithe The t. fow Allice t rabice saw rabbbit. s rabit. sabit T


The generated sample evaluates mechanics, not model quality; this tiny fixture is expected to repeat and misspell text.

## Verify seeded subprocess reproducibility

Running the identical command again starts another process, reloads all files, and should reproduce the exact output.

In [11]:
repeated_generation_result = run_generation_script(generation_arguments)

print(
    "Independent processes match exactly:",
    repeated_generation_result.standard_output == generation_result.standard_output,
)
assert repeated_generation_result.return_code == 0
assert repeated_generation_result.standard_output == generation_result.standard_output

Independent processes match exactly: True


## Preserve a prompt longer than context

The 120-character prompt exceeds the model's 64-token context, but standard output must still begin with the complete prompt.

In [12]:
long_prompt = "Alice " * 20
long_prompt_result = run_generation_script(
    [
        *checkpoint_arguments,
        "--prompt",
        long_prompt,
        "--max-new-tokens",
        "10",
        "--seed",
        "9004",
    ]
)

print("Return code:", long_prompt_result.return_code)
print(
    "Complete prompt preserved:",
    long_prompt_result.standard_output.startswith(long_prompt),
)
assert long_prompt_result.return_code == 0
assert long_prompt_result.standard_output.startswith(long_prompt)

Return code: 0
Complete prompt preserved: True


Only the final 64 prompt tokens influenced the continuation, even though the full prompt remains in the displayed result.

## Reject invalid command-line values

`argparse` rejects a zero temperature before any package is loaded and returns its conventional command-line status code `2`.

In [13]:
invalid_temperature_result = run_generation_script(
    ["--prompt", "Alice", "--temperature", "0"]
)

print("Return code:", invalid_temperature_result.return_code)
print(
    "Error mentions positive value:",
    "value must be positive" in invalid_temperature_result.standard_error,
)
assert invalid_temperature_result.return_code == 2
assert "value must be positive" in invalid_temperature_result.standard_error

Return code: 2
Error mentions positive value: True


## Report package and tokenizer errors cleanly

Runtime loading or encoding failures use status code `1`, print no generated text, and send a concise message to standard error.

In [14]:
missing_package_result = run_generation_script(
    [
        "--checkpoint-path",
        str(temporary_root / "missing_package"),
        "--prompt",
        "Alice",
    ]
)
unknown_character_result = run_generation_script(
    [*checkpoint_arguments, "--prompt", "Alice!", "--max-new-tokens", "5"]
)

print("Missing package return code:", missing_package_result.return_code)
print("Missing package stdout empty:", missing_package_result.standard_output == "")
print(
    "Missing package message:",
    missing_package_result.standard_error.strip(),
)
print("Unknown character return code:", unknown_character_result.return_code)
print(
    "Unknown character message:",
    unknown_character_result.standard_error.strip(),
)

assert missing_package_result.return_code == 1
assert missing_package_result.standard_output == ""
assert "Missing required files" in missing_package_result.standard_error
assert unknown_character_result.return_code == 1
assert unknown_character_result.standard_output == ""
assert "Unknown tokens" in unknown_character_result.standard_error

Missing package return code: 1
Missing package stdout empty: True
Missing package message: error: Missing required files: ['model_state.pt', 'tokenizer.json', 'model_config.json'].
Unknown character return code: 1
Unknown character message: error: Unknown tokens: ['!'].


## Why normal output stays quiet

Successful execution prints only generated text, which supports shell redirection such as `python generate.py ... > output.txt`.

Diagnostics and errors belong on standard error or behind a future `--verbose` option.

This separation also makes the script easier to compose with other command-line tools.

## Run the same commands in a shell

With a persistent package, the equivalent commands are:

```bash
python generate.py     --checkpoint-path saved_tiny_gpt     --prompt "Alice"     --max-new-tokens 80     --temperature 0.8     --top-k 10     --seed 9003
```

```bash
python generate.py --help
```

The prompt is required, while every other option has a default.

## Know when one file stops being minimal

This chapter keeps everything in `generate.py` to prove independence from notebook state.

Duplicating model and tokenizer implementations becomes risky once training and generation scripts evolve separately.

A maintainable next layout would place architecture, tokenizer, and package-loading code in importable modules and keep `generate.py` focused on CLI parsing plus orchestration.

Minimal means the smallest reliable interface, not the fewest lines at any cost.

## Clean up generated script and package files

All subprocess checks finish before the temporary directory is removed.

In [15]:
temporary_directory.cleanup()

print("Temporary workspace removed:", not temporary_root.exists())
assert not temporary_root.exists()

Temporary workspace removed: True


## Verify chapter contracts

The final checks cover compilation, CLI help, quiet success, cross-process determinism, prompt preservation, error status codes, and cleanup.

In [16]:
assert generation_result.return_code == 0
assert generation_result.standard_error == ""
assert repeated_generation_result.standard_output == generation_result.standard_output
assert long_prompt_result.standard_output.startswith(long_prompt)
assert invalid_temperature_result.return_code == 2
assert missing_package_result.return_code == 1
assert unknown_character_result.return_code == 1
assert not temporary_root.exists()
print("All minimal generation-script contracts passed.")

All minimal generation-script contracts passed.


## Common mistakes

- Referencing notebook variables makes the script fail in a fresh process.

- Copying stale architecture or artifact names breaks compatibility with the saved package.

- Trusting unvalidated JSON or unrestricted pickle loading weakens failure handling and safety.

- Resetting global RNG state inside generation makes the function affect unrelated code.

- Returning only the cropped conditioning prefix silently removes the beginning of long prompts.

- Printing loading diagnostics to standard output makes redirection capture noise with generated text.

- Accepting invalid temperature, token counts, or top-k values delays simple CLI errors until model execution.

- Supporting multiple undocumented checkpoint schemas makes a minimal loader ambiguous.

- Duplicating model code across permanent scripts invites architecture drift after this teaching step.

## Takeaways

- A script begins with no notebook objects and must define or load every dependency.

- `argparse` turns one file into a reusable interface without source edits.

- A reliable loader validates package files, tokenizer/config schemas, vocabulary size, state keys, and tensor shapes.

- Restricted CPU loading, strict state restoration, and evaluation mode are appropriate inference defaults.

- Explicit generators provide reproducible sampling without global RNG side effects.

- Subprocess tests prove that the script works outside the notebook kernel.

- Quiet standard output and meaningful exit codes make the script useful in shell workflows.

## What comes next

The one-file script proves the command-line boundary, while shared modules can now remove duplicated architecture and tokenizer code.

That refactor turns a teaching script into the beginning of a maintainable training and inference package.